# RTE external LCA Monte Carlo sources

This notebook builds LCA Monte Carlo runs for the RTE case study and stages them in pyaesa project folders for lette processing by the ASR notebook. 

It reads the two RTE LCA CSV files from the folder set by `LCA_SOURCE_DIR` in the configuration cell.

LCA runs sampling method:

- 2019 to 2025: the deterministic LCA value is repeated for every Monte Carlo run (no background system uncertainty from prospective scenarios)
- 2026 to 2060: background system uncertainty - for each SSP, sample one GMST category uniformly among categories available for that SSP, then sample one `premise` model-scenario pair uniformly within that category.
- Sampling probability of a premise model-scenario pair is:

     `1 / number of GMST categories for the SSP / number of model-scenario pairs in the selected GMST category`.

| SSP scenario | GMST category | Category draw probability |
| --- | --- | --- |
| SSP1 | <1.5 | 20% |
| SSP1 | 1.5-1.7 | 20% |
| SSP1 | 1.7-2.0 | 20% |
| SSP1 | 2.0-2.5 | 20% |
| SSP1 | 2.5-2.8 | 20% |
| SSP2 | <1.5 | 14.29% |
| SSP2 | 1.5-1.7 | 14.29% |
| SSP2 | 1.7-2.0 | 14.29% |
| SSP2 | 2.0-2.5 | 14.29% |
| SSP2 | 2.5-2.8 | 14.29% |
| SSP2 | 2.8-3.0 | 14.29% |
| SSP2 | 3.0-3.2 | 14.29% |
| SSP5 | 1.7-2.0 | 50% |
| SSP5 | >3.5 | 50% | check

## `pyaesa` installation

This notebook uses the `pyaesa` Python package. Install the release from PyPI before running the workflow:

```bash
python -m pip install pyaesa
```

For package documentation, API reference, and tutorials, see [pyaesa.readthedocs.io](https://pyaesa.readthedocs.io/). 

The source code is available on GitHub at [AESAtoolkit/pyaesa](https://github.com/AESAtoolkit/pyaesa).

## Configuration


In [ ]:
"""Build compact external LCA Monte Carlo source files for the RTE publication case."""

from dataclasses import dataclass
import json
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd

from pyaesa import prepare_external_inputs, set_workspace

WORKSPACE_TOP = (
    r"C:\Users\Erwan\Documents\UNCASExt_demo"  # replace with your pyaesa set_workspace path
)
LCA_SOURCE_DIR = Path("lca")
PROJECT_BY_SSP = {
    "SSP2": "rte_scenarios_ssp2",
    "SSP1": "rte_scenarios_ssp1",
    "SSP5": "rte_scenarios_ssp5",
}

HISTORICAL_CSV = LCA_SOURCE_DIR / "rte_lca_historical_2019_2025.csv"
PROSPECTIVE_CSV = LCA_SOURCE_DIR / "rte_lca_prospective_2026_2060.csv"

N_DRAWS = 300_000
RANDOM_SEED = 20260515
CHUNK_RUNS = 10_000
HISTORICAL_YEARS = tuple(range(2019, 2026))
PROSPECTIVE_YEARS = tuple(range(2026, 2061))
RTE_SCENARIOS = (("reference", "m0"), ("reference", "n03"))
SSP_SCENARIOS = tuple(PROJECT_BY_SSP)
SELECTOR_R_C = "FR"
SELECTOR_S_P = "Electricity"
GWP_IMPACT = "GWP_100"
GMST_ORDER = (
    "<1.5",
    "1.5-1.7",
    "1.7-2.0",
    "2.0-2.5",
    "2.5-2.8",
    "2.8-3.0",
    "3.0-3.2",
    "3.2-3.5",
    ">3.5",
)
PAIR_COLUMNS = ["gmst", "iam_model", "iam_scenario"]


@dataclass(frozen=True)
class ScenarioSpec:
    """RTE consumption and production scenario labels used in the input CSVs."""

    consumption: str
    production: str

    @property
    def version_name(self) -> str:
        """Return the RTE version token without SSP suffix."""
        return f"rte_{self.consumption}_{self.production}"

    def ssp_version_name(self, ssp_label: str) -> str:
        """Return the compact external LCA version token for one SSP."""
        return f"{self.version_name}_{ssp_label.lower()}"


@dataclass(frozen=True)
class MethodSpec:
    """External LCA method name and the impact columns assigned to it."""

    name: str
    impacts: tuple[str, ...]


@dataclass(frozen=True)
class SspDraws:
    """Sampled IAM candidate positions reused by every RTE version for one SSP."""

    ssp: str
    candidates: pd.DataFrame
    draw_log: pd.DataFrame
    pair_positions: np.ndarray


## Read input tables

Load the two source CSVs from `LCA_SOURCE_DIR`.


In [ ]:
def read_source_csv(path: Path) -> pd.DataFrame:
    """Read one RTE LCA source table and normalize scalar columns."""
    frame = pd.read_csv(path, keep_default_na=False)
    out = frame.copy()
    for column in [name for name in out.columns if name not in {"year", "value"}]:
        out[column] = out[column].astype(str).str.strip()
    out["year"] = cast(pd.Series, pd.to_numeric(out["year"], errors="raise")).astype("int64")
    out["value"] = cast(pd.Series, pd.to_numeric(out["value"], errors="raise")).astype("float64")
    return out


def read_sources() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read historical and prospective LCA source CSVs from LCA_SOURCE_DIR."""
    historical = read_source_csv(HISTORICAL_CSV)
    prospective = read_source_csv(PROSPECTIVE_CSV)
    return historical, prospective


## Sample IAM model scenario pairs

For each SSP, the notebook samples a GMST category uniformly, then samples one IAM model scenario pair uniformly within that category. The same SSP draw log is reused for both RTE versions.


In [ ]:
def filter_scenario(frame: pd.DataFrame, rte_scenario: ScenarioSpec) -> pd.DataFrame:
    """Select rows for one RTE consumption and production scenario."""
    mask = frame["rte_consumption_scenario"].eq(rte_scenario.consumption) & frame[
        "rte_production_scenario"
    ].eq(rte_scenario.production)
    return frame.loc[mask].copy()


def sort_candidates(candidates: pd.DataFrame) -> pd.DataFrame:
    """Sort IAM candidates by GMST category, model, and scenario."""
    order = {gmst: index for index, gmst in enumerate(GMST_ORDER)}
    out = candidates.copy()
    out["_gmst_order"] = out["gmst"].replace(order)
    return (
        out.sort_values(["_gmst_order", "iam_model", "iam_scenario"], kind="mergesort")
        .drop(columns="_gmst_order")
        .reset_index(drop=True)
    )


def shared_candidates_for_ssp(prospective: pd.DataFrame, selected_ssp: str) -> pd.DataFrame:
    """Return the ordered candidate set used for one SSP."""
    first_consumption, first_production = RTE_SCENARIOS[0]
    rows = filter_scenario(prospective, ScenarioSpec(first_consumption, first_production))
    rows = rows.loc[rows["ssp_scenario"].eq(selected_ssp)]
    return sort_candidates(rows.loc[:, PAIR_COLUMNS].drop_duplicates())


def sample_pair_positions(candidates: pd.DataFrame, rng: np.random.Generator) -> np.ndarray:
    """Draw IAM candidate positions with uniform GMST categories then uniform pairs."""
    categories = [gmst for gmst in GMST_ORDER if gmst in set(candidates["gmst"])]
    category_draws = rng.integers(0, len(categories), size=N_DRAWS, dtype=np.int64)
    positions = np.empty(N_DRAWS, dtype=np.int64)
    for category_index, gmst in enumerate(categories):
        mask = category_draws == category_index
        choices = candidates.index[candidates["gmst"].eq(gmst)].to_numpy(dtype=np.int64)
        positions[mask] = rng.choice(choices, size=int(mask.sum()), replace=True)
    return positions


def build_shared_draws(prospective: pd.DataFrame) -> dict[str, SspDraws]:
    """Build one reusable Monte Carlo draw log per SSP."""
    draws = {}
    for ssp_index, selected_ssp in enumerate(SSP_SCENARIOS):
        candidates = shared_candidates_for_ssp(prospective, selected_ssp)
        rng = np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, ssp_index]))
        positions = sample_pair_positions(candidates, rng)
        selected = candidates.iloc[positions].reset_index(drop=True)
        draw_log = selected.copy()
        draw_log.insert(0, "run_index", np.arange(N_DRAWS, dtype=np.int64))
        draw_log["ssp_scenario"] = selected_ssp
        draws[selected_ssp] = SspDraws(
            selected_ssp,
            candidates,
            draw_log[["run_index", "ssp_scenario", *PAIR_COLUMNS]],
            positions,
        )
    return draws


## Build compact LCA arrays

Historical rows use the deterministic source value. Prospective rows are prepared as a candidate by identity matrix so the Monte Carlo run table can be written with NumPy indexing.


In [ ]:
def impact_units_by_impact(frame: pd.DataFrame) -> dict[str, str]:
    """Map each impact category to its source unit."""
    units = {}
    for impact, group in frame.groupby("impact", sort=False):
        values = tuple(group["impact_unit"].drop_duplicates().tolist())
        units[str(impact)] = str(values[0])
    return units


def build_identity(
    historical: pd.DataFrame,
    prospective: pd.DataFrame,
    selected_ssp: str,
    selected_impacts: tuple[str, ...],
) -> pd.DataFrame:
    """Build the compact external LCA row identity table for one method and SSP."""
    unit_frame = cast(
        pd.DataFrame,
        pd.concat(
            [historical[["impact", "impact_unit"]], prospective[["impact", "impact_unit"]]],
            ignore_index=True,
        ),
    )
    units = impact_units_by_impact(unit_frame)
    rows = [
        {
            "year": year,
            "lca_ssp_scenario": "",
            "r_c": SELECTOR_R_C,
            "s_p": SELECTOR_S_P,
            "impact": impact,
            "impact_unit": units[impact],
        }
        for year in HISTORICAL_YEARS
        for impact in selected_impacts
    ]
    rows.extend(
        {
            "year": year,
            "lca_ssp_scenario": selected_ssp,
            "r_c": SELECTOR_R_C,
            "s_p": SELECTOR_S_P,
            "impact": impact,
            "impact_unit": units[impact],
        }
        for year in PROSPECTIVE_YEARS
        for impact in selected_impacts
    )
    identity = pd.DataFrame(rows)
    identity.insert(0, "public_row_id", np.arange(len(identity), dtype=np.int64))
    return cast(
        pd.DataFrame,
        identity[
            ["public_row_id", "year", "lca_ssp_scenario", "r_c", "s_p", "impact", "impact_unit"]
        ],
    )


def historical_values(
    historical: pd.DataFrame, identity: pd.DataFrame, selected_impacts: tuple[str, ...]
) -> np.ndarray:
    """Return deterministic historical values in compact identity order."""
    source = historical.loc[
        historical["impact"].isin(selected_impacts), ["year", "impact", "value"]
    ].copy()
    lookup = source.set_index(["year", "impact"])["value"]
    historical_identity = identity.loc[identity["year"].isin(HISTORICAL_YEARS), ["year", "impact"]]
    keys = pd.MultiIndex.from_frame(historical_identity)
    values = lookup.reindex(keys)
    return values.to_numpy(dtype=np.float64)


def prospective_matrix(
    prospective: pd.DataFrame,
    candidates: pd.DataFrame,
    identity: pd.DataFrame,
    selected_impacts: tuple[str, ...],
) -> np.ndarray:
    """Build candidate by compact row prospective values for vectorized sampling."""
    columns = identity.loc[identity["year"].isin(PROSPECTIVE_YEARS), ["year", "impact"]].copy()
    columns["column_position"] = np.arange(len(columns), dtype=np.int64)
    pairs = candidates.loc[:, PAIR_COLUMNS].copy()
    pairs["pair_position"] = np.arange(len(pairs), dtype=np.int64)
    source = prospective.loc[
        prospective["impact"].isin(selected_impacts), [*PAIR_COLUMNS, "year", "impact", "value"]
    ].copy()
    positioned = source.merge(pairs, on=PAIR_COLUMNS, how="left", validate="many_to_one")
    positioned = positioned.merge(
        columns, on=["year", "impact"], how="right", validate="many_to_one"
    )
    matrix = np.full((len(pairs), len(columns)), np.nan, dtype=np.float64)
    matrix[
        positioned["pair_position"].to_numpy(dtype=np.int64),
        positioned["column_position"].to_numpy(dtype=np.int64),
    ] = positioned["value"].to_numpy(dtype=np.float64)
    return matrix


## Write pyaesa external LCA sources

Each output folder receives `public_row_identity.csv`, `lca_runs.csv`, `draw_log.csv`, and `generation_metadata.json` in the compact external LCA layout expected by pyaesa.


In [ ]:
def write_lca_runs(
    path: Path, historical: np.ndarray, prospective: np.ndarray, pair_positions: np.ndarray
) -> None:
    """Write compact LCA Monte Carlo runs in chunks to limit memory use."""
    n_columns = int(historical.size + prospective.shape[1])
    columns = [str(index) for index in range(n_columns)]
    first_chunk = True
    for start in range(0, N_DRAWS, CHUNK_RUNS):
        stop = min(start + CHUNK_RUNS, N_DRAWS)
        message = f"\r  writing Monte Carlo runs {start + 1:,} to {stop:,} of {N_DRAWS:,}"
        print(message.replace(",", " "), end="", flush=True)
        values = np.empty((stop - start, n_columns), dtype=np.float64)
        values[:, : historical.size] = historical
        values[:, historical.size :] = prospective[pair_positions[start:stop], :]
        frame = pd.DataFrame(values, columns=columns)
        frame.insert(0, "run_index", np.arange(start, stop, dtype=np.int64))
        frame.to_csv(
            path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
            float_format="%.17g",
        )
        first_chunk = False
    print()


def write_source(
    output_folder: Path,
    historical: pd.DataFrame,
    prospective: pd.DataFrame,
    rte_scenario: ScenarioSpec,
    selected_ssp: str,
    method_spec: MethodSpec,
    draws: SspDraws,
) -> dict[str, object]:
    """Write one compact external LCA source folder for pyaesa."""
    print(f"Preparing {output_folder.name}", flush=True)
    output_folder.mkdir(parents=True, exist_ok=True)
    for filename in (
        "public_row_identity.csv",
        "lca_runs.csv",
        "draw_log.csv",
        "generation_metadata.json",
    ):
        path = output_folder / filename
        if path.exists():
            path.unlink()
    identity = build_identity(historical, prospective, selected_ssp, method_spec.impacts)
    h_values = historical_values(historical, identity, method_spec.impacts)
    p_values = prospective_matrix(prospective, draws.candidates, identity, method_spec.impacts)
    identity.to_csv(output_folder / "public_row_identity.csv", index=False)
    write_lca_runs(output_folder / "lca_runs.csv", h_values, p_values, draws.pair_positions)
    draw_log = draws.draw_log.copy()
    draw_log["rte_consumption_scenario"] = rte_scenario.consumption
    draw_log["rte_production_scenario"] = rte_scenario.production
    draw_log.to_csv(output_folder / "draw_log.csv", index=False)
    metadata = {
        "version_name": output_folder.name.rsplit("__", maxsplit=1)[0],
        "lcia_method": method_spec.name,
        "rte_consumption_scenario": rte_scenario.consumption,
        "rte_production_scenario": rte_scenario.production,
        "ssp_scenario": selected_ssp,
        "n_draws": N_DRAWS,
        "n_public_rows": int(len(identity)),
        "n_iam_pairs": int(len(draws.candidates)),
        "gmst_categories": [
            gmst for gmst in GMST_ORDER if gmst in set(draws.draw_log["gmst"].unique())
        ],
    }
    (output_folder / "generation_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print(f"Completed {output_folder.name}", flush=True)
    return metadata


## Initialize the workspace

`set_workspace(...)` selects the pyaesa workspace and `prepare_external_inputs(...)` creates the external LCA staging folders for the three SSP projects.


In [ ]:
set_workspace(WORKSPACE_TOP)
for project_name in PROJECT_BY_SSP.values():
    prepare_external_inputs(project_name=project_name)


## Generate all RTE external LCA sources

Run this cell once to stage both RTE versions, all SSPs, and both LCIA methods. It prints the active source folder and Monte Carlo chunk ranges while writing `lca_runs.csv`, so progress remains visible during the long generation step.


In [ ]:
print("Loading RTE LCA source CSVs")
historical_source, prospective_source = read_sources()
impact_names = tuple(dict.fromkeys(historical_source["impact"].tolist()))
lca_methods = (
    MethodSpec("pb_lcia", tuple(impact for impact in impact_names if impact != GWP_IMPACT)),
    MethodSpec("gwp100_lcia", (GWP_IMPACT,)),
)
print("Sampling IAM model scenario pairs for each SSP")
draws_by_ssp = build_shared_draws(prospective_source)
manifest = []

for consumption_label, production_label in RTE_SCENARIOS:
    rte_scenario_spec = ScenarioSpec(consumption_label, production_label)
    scenario_historical = filter_scenario(historical_source, rte_scenario_spec)
    scenario_prospective = filter_scenario(prospective_source, rte_scenario_spec)
    for current_ssp in SSP_SCENARIOS:
        project_name = PROJECT_BY_SSP[current_ssp]
        ssp_prospective = scenario_prospective.loc[
            scenario_prospective["ssp_scenario"].eq(current_ssp)
        ].copy()
        version_name = rte_scenario_spec.ssp_version_name(current_ssp)
        for lca_method_spec in lca_methods:
            source_folder = (
                Path(WORKSPACE_TOP)
                / "pyaesa"
                / project_name
                / "A_lca"
                / "external_lca"
                / "monte_carlo"
                / f"{version_name}__{lca_method_spec.name}"
            )
            print(f"Writing {source_folder.name} in {project_name}")
            manifest.append(
                write_source(
                    source_folder,
                    scenario_historical,
                    ssp_prospective,
                    rte_scenario_spec,
                    current_ssp,
                    lca_method_spec,
                    draws_by_ssp[current_ssp],
                )
            )

for project_name in PROJECT_BY_SSP.values():
    manifest_path = (
        Path(WORKSPACE_TOP)
        / "pyaesa"
        / project_name
        / "A_lca"
        / "external_lca"
        / "monte_carlo"
        / "rte_lca_generation_manifest.json"
    )
    project_entries = [entry for entry in manifest if entry["ssp_scenario"].lower() in project_name]
    manifest_path.write_text(
        json.dumps({"n_draws": N_DRAWS, "sources": project_entries}, indent=2), encoding="utf-8"
    )
print(f"Prepared {len(manifest)} compact external LCA sources.")
